# **Notebook 7: Pipeline Yolo V9**

*This notebook is used for round 1, round 2 and the testing tiles.*

This Jupyter notebook runs the pipeline to train, test and predict parking spaces using Yolo V9

The script was written for a project commissioned by the Municipality of Wageningen in relation to the course Remote Sensing and GIS integration (GRS60312).

The authors of the script are Polly Cheung, Pascal Dubbelman, Anthony Jansen, Iris Lagemaat, and Susanna van de Wetering.

*Last edited: 23/06/2026*

First, install ultralytics in order to use pre-trained YOLO object detection models and install plug-ins

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
from pathlib import Path
import sys
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


**Step 1:** Define paths

In [10]:
# Point to project root.
base = Path('/content/gdrive/MyDrive/ParkingSpaceDetection')

# Directories training and validation split.
train_val_tif = base / 'data' / 'source_train_val' / 'tif_tiles'
train_val_xml = base / 'data' / 'source_train_val' / 'xml_labels'
train_val_png = base / 'data' / 'source_train_val'/ 'png_tiles'
train_val_txt = base / 'data' / 'source_train_val' / 'txt_labels'

train_png = base / 'data' / 'images' / 'train'
val_png   = base / 'data' / 'images' / 'val'
train_txt = base / 'data' / 'labels' / 'train'
val_txt   = base / 'data' / 'labels' / 'val'

# Directories test split.
test_tif = base / 'data' / 'source_test' / 'test_tiles'
test_xml = base / 'data' / 'source_test' / 'test_labels'
test_png = base / 'data' / 'images' / 'test'
test_txt = base / 'data' / 'labels' / 'test'

# Directories prediction split.
predict_tif = base / 'data' / 'source_pred'
predict_png = base / 'data' / 'images' / 'predict'

geojson_output_path = base / 'results' / 'predictionsYoloV9.geojson'
yaml_path = base / 'data.yaml'
yolo9_output = base / 'yolo11_output'

**Step 2:** Convert the `.tif` files to `.png`

In [ ]:
sys.path.append(str(base))

from python.01_tif_to_png import tif_to_png

# Convert .tif to .png.
tif_to_png(train_val_tif, train_val_png)

KeyboardInterrupt: 

**Step 3:** Convert `.xml` files to `.txt` files

In [ ]:
sys.path.append(str(base))

from python.02_xml_to_txt import xml_to_txt

# Convert .xml to .txt.
xml_to_txt(train_val_xml, train_val_txt)

**Step 4:** Create training and validation data

In [ ]:
sys.path.append(str(base))

from python.03_train_val_data_split import create_dirs, split_data, move_pairs

# Create directories for training and validation data.
create_dirs(train_png, val_png, train_txt, val_txt)

# Split data into 80% train, 20% val.
train_imgs, val_imgs = split_data(train_val_png)

# Move training and validation data into directories.
move_pairs(train_imgs, train_val_txt, train_png, train_txt)
move_pairs(val_imgs,   train_val_txt, val_png,   val_txt)

**Step 5:** Train model

In [ ]:
sys.path.append(str(base))

from ultralytics import YOLO
from python.04_train_model import train_model

# Model.
yolo = YOLO("yolov9c.pt")

# Model training.
train_model(yolo, yaml_path, yolo9_output )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/MyDrive/ParkingSpaceDetection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): RepNCSPELAN4(
        (cv1): Conv(
          (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Sequential(
          (0): RepCSP(
            (cv1): Conv(
              (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
              

**Step 6:** Validate model

In [5]:
# Load best trained model
model = YOLO('/content/gdrive/MyDrive/ParkingSpaceDetection/yolo9_output/train/weights/best.pt')

# Validate the loaded 'best.pt' model on the validation dataset
validation_results = model.val(data=yaml_path)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv9c summary (fused): 156 layers, 25,320,019 parameters, 0 gradients, 102.3 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 1.9±0.5 MB/s, size: 1393.3 KB)
val: Scanning /content/gdrive/MyDrive/ParkingSpaceDetection/data/labels/val.cache... 35 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 35/35 6.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 7.7s/it 23.1s
                   all         35        663      0.896      0.792      0.857      0.514
Speed: 7.1ms preprocess, 52.3ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /content/runs/detect/val


**Step 7:** Test model on 'unseen' data

In [6]:
# Validate on test split
test_val = model.val(data=yaml_path, split='test')

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 1.8±0.2 MB/s, size: 1220.0 KB)
val: Scanning /content/gdrive/MyDrive/ParkingSpaceDetection/data/labels/test.cache... 40 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40 12.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 7.8s/it 23.4s
                   all         40        494      0.522       0.37      0.318      0.136
Speed: 10.5ms preprocess, 39.7ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val-2


**Step 8:** Let the model predict on neighbourhood

In [7]:
# Run prediction
predict_results = model.predict(source=predict_png, save=True, save_txt=True, save_conf=True, conf=0.3)


image 1/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_1.png: 1024x1024 9 items, 21.0ms
image 2/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_10.png: 1024x1024 17 items, 21.2ms
image 3/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_100.png: 1024x1024 (no detections), 21.0ms
image 4/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_101.png: 1024x1024 19 items, 21.1ms
image 5/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_102.png: 1024x1024 31 items, 21.2ms
image 6/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_103.png: 1024x1024 2 items, 21.0ms
image 7/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_104.png: 1024x1024 (no detections), 21.2ms
image 8/241 /content/gdrive/MyDrive/ParkingSpaceDetection/data/images/predict/predict_105.png: 1024x1024 27 items, 21.2ms
image 9/241 /c

**Step 9:** Georeference the predicted bounding boxes.

In [ ]:
sys.path.append(str(base))

from python.05_georeference_BB import georeference_BB

georeference_BB(predict_results, predict_tif, geojson_output_path)

Successfully saved 1575 bounding box predictions to /content/gdrive/MyDrive/ParkingSpaceDetection/results/predictionsYoloV9.geojson
